In [1]:
import pandas as pd

df = pd.read_csv('../../datasets/MX_with_labels.csv')
df.head()

#print(df[df["name"] == "APT."])

# print(df['popularity'].value_counts())
# df['popularity'].value_counts().plot(kind='bar')
# plt.title('Popularity Distribution')

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,5Z75FvRFiultmPFWHx5jQ7,7 Dias,"Gabito Ballesteros, Tito Double P",1,0,1,MX,2025-02-17,85,True,...,-3.749,1,0.0614,0.3860,0.000000,0.0839,0.577,111.913,3,Higher
1,78HEzDEs1QUnHB2DbxgC1s,Te Quería Ver,"Alemán, Neton Vega",2,1,1,MX,2025-02-17,82,False,...,-5.182,0,0.0681,0.1880,0.000017,0.0922,0.448,100.019,4,About_Average
2,0LTwdL5yZ6YOTEGUQPFuSN,ROSONES,Tito Double P,3,1,1,MX,2025-02-17,88,True,...,-5.939,1,0.0318,0.7040,0.000010,0.1170,0.604,120.129,3,Lower
3,7sd6zMrgGpEa7NkQm9TRrg,NADIE,Tito Double P,4,1,2,MX,2025-02-17,87,True,...,-4.710,1,0.1140,0.4650,0.000000,0.1200,0.526,92.604,4,About_Average
4,4eLDmhsJW3JoZTXCAozHor,Loco,Neton Vega,5,-3,45,MX,2025-02-17,61,False,...,-5.502,1,0.0686,0.0741,0.007680,0.1390,0.636,91.981,4,Lower


In [2]:

from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import RidgeClassifier

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'is_explicit',
       'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [3]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    RidgeClassifier()
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


print("Ridge Regression")
mean_squared_error(y_test, y_pred)


Ridge Regression


105.1667366652667

In [4]:
pred = pd.DataFrame(y_pred).value_counts()
test = pd.DataFrame(y_test).value_counts()

print(pred.describe())
print(test.describe())

count     19.000000
mean     250.631579
std      312.779868
min        3.000000
25%       30.500000
50%      171.000000
75%      226.000000
max      987.000000
Name: count, dtype: float64
count     56.000000
mean      85.035714
std      118.018175
min        1.000000
25%        3.500000
50%       27.500000
75%      104.750000
max      391.000000
Name: count, dtype: float64


In [5]:
hyperParameters = {'ridgeclassifier__alpha':[150000, 200000, 250000, 300000, 350000, 400000, 450000, 500000, 550000, 600000, 650000, 700000, 750000, 800000, 850000, 900000, 950000]}

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

RRGrid = GridSearchCV(
    pipeline,
    param_grid=hyperParameters,
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    verbose=3,
)

RRGrid.fit(X_train, y_train)
print("Best alpha: ", RRGrid.best_params_["ridgeclassifier__alpha"])
print("Best score: ", RRGrid.best_score_)

Fitting 5 folds for each of 17 candidates, totalling 85 fits
[CV 1/5] END ...ridgeclassifier__alpha=150000;, score=-91.237 total time=   0.0s
[CV 2/5] END ...ridgeclassifier__alpha=150000;, score=-79.566 total time=   0.0s
[CV 3/5] END ...ridgeclassifier__alpha=150000;, score=-83.955 total time=   0.0s
[CV 4/5] END ...ridgeclassifier__alpha=150000;, score=-75.249 total time=   0.0s
[CV 5/5] END ...ridgeclassifier__alpha=150000;, score=-73.882 total time=   0.0s
[CV 1/5] END ...ridgeclassifier__alpha=200000;, score=-91.237 total time=   0.0s
[CV 2/5] END ...ridgeclassifier__alpha=200000;, score=-79.566 total time=   0.0s
[CV 3/5] END ...ridgeclassifier__alpha=200000;, score=-83.955 total time=   0.0s
[CV 4/5] END ...ridgeclassifier__alpha=200000;, score=-75.249 total time=   0.0s
[CV 5/5] END ...ridgeclassifier__alpha=200000;, score=-73.882 total time=   0.0s
[CV 1/5] END ...ridgeclassifier__alpha=250000;, score=-91.237 total time=   0.0s
[CV 2/5] END ...ridgeclassifier__alpha=250000;, 